# Assignment 2: Preparation and Testing

In order to perform Assignment 2 in the course, I will use the local PostgreSQL connection and set of queries, checking for the hypotheses I've listed.

In [19]:
from sqlalchemy import create_engine
from sqlalchemy import text
from dotenv import load_dotenv
import os
import pandas as pd
import plotly.express as px

## 1. Create a connection to local database:

In [5]:
load_dotenv()
db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USER")

if not db_name or not db_user:
    raise ValueError("Error: DB_NAME or DB_USER not found in environment variables. Check your .env file!")

engine = create_engine(f"postgresql://{db_user}@localhost/{db_name}")
print("SQLalchemy engine was created and can be used.")

SQLalchemy engine was created and can be used.


## 2. Test a query:

In [16]:
query_test = """SELECT g.YEAR as year, COUNT(DISTINCT t.team_id) AS team_count
FROM team AS t
JOIN athlete_games AS ag ON t.team_id=ag.team_id
JOIN game AS g ON ag.game_id=g.game_id
GROUP BY g.YEAR
ORDER BY g.YEAR;"""

In [10]:
with engine.connect() as connection:
    result = connection.execute(
        text(query_test)
    )
    
    # Iterate through rows
    for row in result:
        print(f"Year: {row[0]}, Athletes: {row[1]}")

Year: 1896, Athletes: 20
Year: 1900, Athletes: 212
Year: 1904, Athletes: 86
Year: 1906, Athletes: 55
Year: 1908, Athletes: 75
Year: 1912, Athletes: 105
Year: 1920, Athletes: 72
Year: 1924, Athletes: 96
Year: 1928, Athletes: 87
Year: 1932, Athletes: 74
Year: 1936, Athletes: 133
Year: 1948, Athletes: 138
Year: 1952, Athletes: 160
Year: 1956, Athletes: 150
Year: 1960, Athletes: 199
Year: 1964, Athletes: 201
Year: 1968, Athletes: 145
Year: 1972, Athletes: 157
Year: 1976, Athletes: 126
Year: 1980, Athletes: 112
Year: 1984, Athletes: 179
Year: 1988, Athletes: 211
Year: 1992, Athletes: 240
Year: 1994, Athletes: 101
Year: 1996, Athletes: 246
Year: 1998, Athletes: 108
Year: 2000, Athletes: 243
Year: 2002, Athletes: 115
Year: 2004, Athletes: 258
Year: 2006, Athletes: 114
Year: 2008, Athletes: 293
Year: 2010, Athletes: 117
Year: 2012, Athletes: 245
Year: 2014, Athletes: 120
Year: 2016, Athletes: 249


In [17]:
df_test = pd.read_sql(text(query_test),con=engine)

In [21]:
fig = px.line(df_test, x='year', y='team_count', title='Number of Olympic Teams (1896-2016)', markers=True)
fig.show()

## 3. Query Development
Now the task is to automate the process: to create a set of queries --> to run them all in one Python script --> to create a visualization for each of them. The number of queries corresponds to the following list of assumptions:

### Assumptions to test
> **Assumption 1**: Female participation rates and medal achievements have experienced a sharp upward inflection point in the latter half of the 20th century, particularly following institutional policy changes in international sports federations.<br>

Possible query:
```SQL
-- ratio of women over years:
SELECT
    g.YEAR AS year,
    --COUNT(DISTINCT CASE WHEN a.sex = 'F' THEN a.athlete_id END) AS women_count,
    --COUNT(DISTINCT a.athlete_id) AS total_participants,
    (COUNT(DISTINCT CASE WHEN a.sex = 'F' THEN a.athlete_id END) * 100.0) / COUNT(DISTINCT a.athlete_id) AS share_of_women
FROM athlete AS a
JOIN athlete_games AS ag ON a.athlete_id = ag.athlete_id
JOIN game AS g ON ag.game_id = g.game_id
GROUP BY g.YEAR
ORDER BY g.YEAR;
```

> **Assumption 2**: Certain countries maintain a persistent specialization advantage in specific event categories (e.g., endurance vs. strength sports) that remains stable across decades regardless of geopolitical shifts.<br>
```SQL 
-- meadals per country over years
SELECT
	r.region AS country,
	g.YEAR AS YEAR,
	COUNT(ra.medal_type) AS medals_count
FROM region AS r
JOIN team AS t ON r.noc_id=t.noc_id
JOIN athlete_games AS ag ON t.team_id=ag.team_id
JOIN result_award AS ra on ag.participation_id=ra.participation_id
JOIN game AS g ON ag.game_id=g.game_id
GROUP BY r.region, g.YEAR
ORDER BY g.YEAR;
``` 

> **Assumption 2a** (added): A small elite group of countries (e.g., USA, Soviet Union/Russia, Germany, UK) captures a disproportionately high percentage (e.g., over 50%) of all medals awarded historically.<br>

> **Assumption 3** (re-formulated): While the overall average age of all Olympians hovers consistently around 24–26 years old, specific sports like Gymnastics and Figure Skating have drastically younger average ages (teens) compared to Equestrian or Sailing (late 30s+).<br>

Later I've added a couple more:
> **Assumption 4**: Across physical sports (like basketball, swimming, and rowing), the average height and weight of athletes have grown steadily over the decades due to professionalization, training standards, and nutrition.<br>

> **Assumption 5**: The total number of unique countries/regions (region via noc_regions) participating in the Olympics has grown significantly, reflecting geopolitical shifts (such as post-WWII and post-Soviet independence waves).<br>

> **Assumprion 6**: The total number of distinct events (event) and sports (sport) has expanded aggressively, particularly in the Summer Games, through the introduction of new disciplines and weight classes.<br>

> **Assumption 7**: Summer Games account for the vast majority of total rows, unique athletes, and participating nations compared to Winter Games, which remain structurally smaller in scale.<br>

> **Assumption 8**: Host nations experience a temporary, statistically significant surge in medal production (especially gold medals) during the specific year they host the Games.<br>

> **Assumption 9**: The vast majority of Olympic athletes participate in only a single Olympic Games, while multi-game Olympians (competing in 3+ games) represent a tiny elite minority.<br>